In [1]:
import geopandas as gpd
geojson_pozo = gpd.read_file("./assets/Pozos.geojson")

In [ ]:
cloro = geojson_pozo[geojson_pozo["AÑO"] == "2015"]

In [ ]:
cloro

,CVEGEO_LOC,AÑO,NOM_MUN,NOM_LOC,Fuente de abastecimiento,Coliformes Totales (Ausencia o Presencia/100mL),E. Coli (Ausencia o Presencia/100mL),Arsenico (mg/L),Bario (mg/L),Cadmio (mg/L),...,Nitratos (mg/L),Nitritos (mg/L),Ph,SDT (mg/L),Sulfatos (mg/L),Cloro Total (mg/L),Conductividad (muS/cm),Temperatura (°C),ID,geometry
1,130010001,2015,Acatlán,Acatlán,Pozo Los Fresnos,Presencia,Ausencia,0,0,No hay dato,...,0.4,0.003,6.32,345,0,0,632,24.7,Acatlán_Acatlán_Pozo Los Fresnos,POINT (-98.44139 20.14585)
5,130010021,2015,Acatlán,San Dionisio,Pozo San Dionisio,Presencia,Ausencia,0,0,No hay dato,...,0.9,0.005,6.38,267.3,2,0,380,24.7,Acatlán_San Dionisio_Pozo San Dionisio,POINT (-98.46799 20.12532)
21,130020004,2015,Acaxochitlán,Cuaunepantla,Pozo Cuaunepantla,Presencia,Ausencia,0,0,No hay dato,...,0.4,0.008,7.4,74.5,10,0,118.21,22.4,Acaxochitlán_Cuaunepantla_Pozo Cuaunepantla,POINT (-98.23049 20.15534)
24,130020021,2015,Acaxochitlán,Tlamimilolpa,Manantial Aua Linda,Presencia,Ausencia,0,0,No hay dato,...,0.5,0.017,7.4,96.4,0,0,110.14,18.2,Acaxochitlán_Tlamimilolpa_Manantial Aua Linda,POINT (-98.21528 20.12581)
30,130030001,2015,Actopan,Actopan,Pozo La Noria,Presencia,Ausencia,0,0,No hay dato,...,1.1,0.268,6.44,959,351,0,2.33,22.4,Actopan_Actopan_Pozo La Noria,POINT (-98.95776 20.27231)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
900,1308000080028800,2015,Yahualica,Hueyactétl,Tanque de Distribución,Presencia,Presencia,0,0,No hay dato,...,0.1,0.002,8.2,306.9,3.1,0.1,455,30.8,Yahualica_Hueyactétl_Tanque de Distribución,POINT (-98.40041 20.96568)
912,130810001,2015,Zacualtipán de Ángeles,Zacualtipán,Pozo Chiras,Presencia,Ausencia,0,0,No hay dato,...,1,0,7.4,146.1,0,0,231.13,20.1,Zacualtipán de Ángeles_Zacualtipán_Pozo Chiras,POINT (-98.65909 20.64167)
913,130810001,2015,Zacualtipán de Ángeles,Zacualtipán,Pozo Oscura,Presencia,Ausencia,0,0,No hay dato,...,0.5,0,7.6,87.6,0,0.02,146.7,19.3,Zacualtipán de Ángeles_Zacualtipán_Pozo Oscura,POINT (-98.65853 20.64822)
930,130840001,2015,Zimapán,Zimapán,Pozo 1,Presencia,Ausencia,0.013,0,No hay dato,...,1.1,0.003,7.1,187,19,0,480,24.8,Zimapán_Zimapán_Pozo 1,POINT (-99.37727 20.73636)


# Municipal filtro

In [84]:
import geopandas as gpd
import numpy as np

df = gpd.read_file("./assets/Acciones_de_desinfeccion_municipal.geojson").drop(columns=["geometry"], errors='ignore')

In [85]:
df = df[df["NOM_MUN"] == "Mineral de la Reforma"]
cloro = df.loc[:, 'CLORO_2020':'CLORO_2024']

In [86]:
cloro = cloro.melt(
    value_vars=cloro.loc[:, "CLORO_2020":"CLORO_2024"].columns,
    var_name="Año",
    value_name="Cloro Libre Residual"
)

In [87]:
cloro["Año"] = (
    cloro["Año"]
    .str.replace("CLORO_", "", regex=False)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

cloro["limite"] = np.where(
    (cloro["Cloro Libre Residual"] >= 0.2) & (cloro["Cloro Libre Residual"] <= 1.5),
    "Limite permisible",
    "Fuera del limite permisible"
)

cloro["Cloro Libre Residual"] = cloro["Cloro Libre Residual"].astype(str)
cloro["limite"] = cloro["Cloro Libre Residual"].replace("-1.0", "No hay dato")
cloro["Cloro Libre Residual"] = cloro["Cloro Libre Residual"].replace("-1.0", "No hay dato")

In [88]:
cloro

,Año,Cloro Libre Residual,limite
0,2020,1.15,1.15
1,2021,No hay dato,No hay dato
2,2022,No hay dato,No hay dato
3,2023,0.37,0.37
4,2024,0.91,0.91


In [52]:
dosificadores = df.loc[:, 'Dosificadores_localidad':'Dosificadores_gasto_agua']

In [53]:
import pandas as pd
cols = dosificadores.loc[:, "Dosificadores_localidad":"Dosificadores_gasto_agua"].columns

# Separar filas (equivalente a separate_rows)
for col in cols:
    dosificadores[col] = dosificadores[col].str.split(",")

dosificadores = dosificadores.explode(cols.tolist())

# Quitar espacios extra (equivalente a str_squish)
for col in cols:
    dosificadores[col] = (
        dosificadores[col]
        .str.strip()                 # quita espacios inicio/fin
        .str.replace(r"\s+", " ", regex=True)  # reduce múltiples espacios a uno
    )

C:\Users\SIGEH\AppData\Local\Temp\ipykernel_14656\329659238.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dosificadores[col] = dosificadores[col].str.split(",")


In [55]:
dosificadores = dosificadores.rename(columns={"Dosificadores_localidad":"Localidad", "Dosificadores_locacion":"Locación", "Dosificadores_anios":"Año", "Dosificadores_marca":"Marca", "Dosificadores_gasto_agua":"Gasto de agua"})

In [56]:
dosificadores

,Localidad,Locación,Año,Marca,Gasto de agua
34,La Victoria,Pozo La Victoria,2020,Milton Roy: PD041-0828-NI,7
34,Palo Gordo,Pozo Palo Gordo,2020,Milton Roy: PD041-0828-NI,6
34,Metepec,Pozo Nuevo Metepec,2023,EMEC: Pompa VCO 1804,6
34,Temaxcalillos,Pozo Temaxcalillos,2023,EMEC: Pompa VCO 1804,7
